### QA with tracing and logging
Every `ask()` call produces an **MLflow trace**: retrieved chunks, prompt, answer, tokens, latency per step.

Every call is also written to `gold.qa_log` (usage, latency, cost).

In [0]:
 %pip install -q databricks-vectorsearch openai "mlflow>=3"

In [0]:
%restart_python

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("gold_schema", "f1_rag_gold")

In [0]:
catalog = dbutils.widgets.get("catalog")
gold_schema = dbutils.widgets.get("gold_schema")
QA_LOG_TABLE = f"{catalog}.{gold_schema}.qa_log"

In [0]:
import os
import sys
import uuid
from datetime import datetime, timezone

sys.path.append(os.path.abspath(".."))

import mlflow
from rag.config import VS_INDEX
from rag.qa import ask

USER = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{USER}/lab12-f1-rag")

### Logging

In [0]:
def ask_and_log(question, category=None, use_retrieval=True):
    r = ask(question, category=category, use_retrieval=use_retrieval)

    row = {
        "request_id": str(uuid.uuid4()),
        "requested_at": datetime.now(timezone.utc),
        "user_name": USER,
        "question": r["question"],
        "category_filter": r["category"],
        "use_retrieval": r["use_retrieval"],
        "index_name": VS_INDEX if r["use_retrieval"] else None,
        "retrieved_chunk_ids": r["chunk_ids"],
        "answer": r["answer"],
        "model": r["model"],
        "prompt_tokens": r["prompt_tokens"],
        "completion_tokens": r["completion_tokens"],
        "latency_ms": r["latency_ms"]
    }
    log_schema = spark.table(QA_LOG_TABLE).schema

    (
        spark.createDataFrame([row], schema=log_schema)
            .write
            .mode("append")
            .saveAsTable(QA_LOG_TABLE)
    )

    return r

In [0]:
def show(r):
    mode = "RAG" if r["use_retrieval"] else "no retrieval"
    print(f"Q: {r['question']}  [{mode}, category={r['category']}]")
    print(f"A: {r['answer']}")
    for title, section, url in r["sources"]:
        print(f"- {title} at {section}  {url}")
    print(f"\t{r['prompt_tokens']} prompt + {r['completion_tokens']} completion tokens, {r['latency_ms']:.0f} ms\n")

### Demo
1. A factual question with RAG.
2. The same question restricted to one category (metadata filter).
3. The same question without retrieval (baseline).
4. A question the articles cannot answer — the assistant should say it does not know.

In [0]:
q = "Which team did Lewis Hamilton join in 2025?"

show(ask_and_log(q))
show(ask_and_log(q, category="driver"))
show(ask_and_log(q, use_retrieval=False))
show(ask_and_log("What is Lewis Hamilton's favourite food?", category="driver"))

### Monitoring
Traces: Experiments → lab12-f1-rag → Traces.

In [0]:
display(spark.sql(f"""
    SELECT use_retrieval,
           COUNT(*) AS requests
           , ROUND(AVG(prompt_tokens)) AS avg_prompt_tokens
           , ROUND(AVG(completion_tokens)) AS avg_completion_tokens
           , ROUND(AVG(latency_ms)) AS avg_latency_ms
    FROM {QA_LOG_TABLE}
    GROUP BY use_retrieval
"""))

In [0]:
display(spark.sql(f"""
    SELECT requested_at
        , question
        , category_filter
        , use_retrieval
        , size(retrieved_chunk_ids) AS chunks
        , LEFT(answer, 120) AS answer
        , latency_ms
    FROM {QA_LOG_TABLE}
    ORDER BY requested_at DESC
    LIMIT 10
"""))